In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

from Bio import SeqIO
import gzip
import re

In [ ]:
# run in shell with ribopy installed
for f in 'S17T0-1' 'S17T0-2' 'S17T24-1' 'S17T24-2' 'S17T72-1' 'S17T72-2' 'S17T72-3' ; do echo $f ;
ribopy dump coverage /path/to/all.ribo ${f} --format tsv -o S17/${f}_coverage.csv.gz
done

## Age-dependent pause sites

In [ ]:
def read_coverage(exp_file: dict, gene_list, buffer=20):
    gene_coverage = []
    
    for exp, file in exp_file.items():
        print(exp)
        coverage = pd.read_csv(file, sep='\t', header=None)
        # get gene_id
        coverage['GeneID'] = coverage[0].str.extract(r'(ENSG\d+)')[0]
        # keep only in gene list
        coverage = coverage[coverage['GeneID'].isin(gene_list.GeneID)]
        for ind, row in coverage.iterrows():
            transcript = pd.DataFrame(np.array(row[1].split(',')).astype(int), columns=['count']).reset_index(names='position')
            if (transcript['count'] != 0).any():
                # keep only CDS region
                match = re.search(r'CDS:([^|]+)', row[0])
                start, end = match.group(1).split('-')
                start, end = int(start) - 1, int(end)
                transcript.query('position >= @start & position < @end', inplace=True)
                transcript = transcript.reset_index(drop=True)
                # group and sum counts of codons
                transcript['codon'] = transcript.index // 3  # 0, 1, 2 -> codon 0
                transcript = transcript.groupby(['codon']).agg({
                    'count': 'sum',
                    'position': 'min',
                }).reset_index()
                # exclude first and last buffer codons
                last = transcript.codon.max()
                transcript = transcript.iloc[buffer:last-buffer]
                
                gene_coverage.append(transcript.assign(
                        gene=row['GeneID'],
                        raw_name=exp,
                        experiment=exp[:-2],
                ))
    
    gene_coverage = pd.concat(gene_coverage)
    return gene_coverage.groupby(['experiment', 'gene', 'codon']).agg({
                    'count': 'sum',
                    'position': 'min',
                }).reset_index()

def calculate_pause_scores(gene_coverage):
    '''Return new dataframe with pause score calculated for each codon, gene, experiment.
    
    Since pause scores are determined per transcript and experiment, can determine them all at once'''
    result = gene_coverage.copy()
    
    # calculate pause_score
    result['pause_score'] = result['count'] / result.groupby(['gene', 'experiment'])['count'].transform('mean')
    return result[result['count'] != 0]

def identify_age_dependent_pauses(df, young_exp, old_exp, mid_exp, min_coverage=0.5, min_total_reads=64, only_significant=True):
    """
    Identifies age-dependent ribosome pause sites based on the provided criteria.

    Args:
        df (pd.DataFrame): DataFrame with columns 'gene', 'experiment', 'codon', and 'count'.
        young_exp (str): Experiment name for young samples.
        old_exp (str): Experiment name for old samples.
        mid_exp (str): Experiment name for intermediate-aged samples.
        min_coverage (float): Minimum average sequencing coverage per codon.
        min_total_reads (int): Minimum total reads in each replicate.

    Returns:
        pd.DataFrame: DataFrame containing identified age-dependent pause sites.
    """

    # Calculate average coverage per codon
    df['coverage'] = df['count'] / 3
    gene_coverage = df.groupby(['gene', 'experiment'])['coverage'].mean().unstack()
    valid_genes = gene_coverage.mean(axis=1)[(gene_coverage.mean(axis=1) >= min_coverage)].index

    # Filter genes with sufficient coverage and total reads
    df_filtered = df[df['gene'].isin(valid_genes)]
    gene_total_reads = df_filtered.groupby(['gene', 'experiment'])['count'].sum().unstack()
    valid_genes = gene_total_reads.min(axis=1)[(gene_total_reads.min(axis=1) >= min_total_reads)].index
    df_filtered = df_filtered[df_filtered['gene'].isin(valid_genes)]

    results = []
    for gene, gene_df in df_filtered.groupby('gene'):
        gene_codons = gene_df[['codon', 'position']].drop_duplicates()
        
        for _, (codon, position) in gene_codons.iterrows():
            young_reads = gene_df[(gene_df['experiment'] == young_exp) & (gene_df['codon'] == codon)]['count'].sum()
            old_reads = gene_df[(gene_df['experiment'] == old_exp) & (gene_df['codon'] == codon)]['count'].sum()

            young_total = gene_df[gene_df['experiment'] == young_exp]['count'].sum() - young_reads
            old_total = gene_df[gene_df['experiment'] == old_exp]['count'].sum() - old_reads

            contingency_table = [[old_reads, young_reads], [old_total, young_total]]
            odds_ratio, p_value = fisher_exact(contingency_table, alternative='two-sided')

            results.append({
                'gene': gene,
                'codon': codon,
                'position': position,
                'young_reads': young_reads,
                'old_reads': old_reads,
                'young_total': young_total,
                'old_total': old_total,
                'odds_ratio': odds_ratio,
                'p_value': p_value
            })

    results_df = pd.DataFrame(results)

    # Benjamini-Hochberg correction
    results_df['adjusted_p_value'] = multipletests(results_df['p_value'], method='fdr_bh')[1]

    # Filter based on criteria
    pause_sites = results_df[
        (results_df['odds_ratio'] < float('inf')) &
        (results_df['old_reads'] + results_df['young_reads'] > 0)
    ]
    if only_significant:
        pause_sites = pause_sites[
            (pause_sites['adjusted_p_value'] < 0.05) &
            (pause_sites['odds_ratio'] > 1)
        ]

    # Check pause score in intermediate sample
    if mid_exp:
        mid_exp_data = df_filtered[df_filtered['experiment'] == mid_exp]
        mid_exp_data = mid_exp_data[mid_exp_data.set_index(['gene', 'codon']).index.isin(pause_sites.set_index(['gene', 'codon']).index)]
        grouped_mid_exp = mid_exp_data.groupby(['gene', 'codon'])['count'].sum().reset_index()
        grouped_mid_exp.rename(columns={'count': 'mid_reads'}, inplace=True)
        pause_sites = pause_sites.merge(grouped_mid_exp, on=['gene', 'codon'], how='left')
        pause_sites['mid_reads'].fillna(0, inplace=True)  # Fill NaNs with 0 if there are no matching rows
    
        pause_sites = pause_sites[pause_sites['old_reads'] > pause_sites['mid_reads']]

    # Check reads in oldest sample vs average
    gene_avg_reads = df_filtered.groupby(['gene'])['count'].mean().reset_index()
    pause_sites = pd.merge(pause_sites, gene_avg_reads, on='gene')
    pause_sites = pause_sites[pause_sites['old_reads'] > pause_sites['count']]

    pause_sites = pause_sites[['gene', 'codon', 'position', 'old_reads', 'young_reads', 'odds_ratio', 'adjusted_p_value']]
    pause_sites['log2_odds_ratio'] = np.log2(pause_sites['odds_ratio'])
    pause_sites['-log10_adjusted_p_value'] = -np.log10(pause_sites['adjusted_p_value'])

    return pause_sites

# add codons to the pause_site df
def add_codons(pause_sites):
    ensg = re.compile(r'(ENSG\d+)')
        
    with gzip.open('/path/to/appris_human_v2_selected.fa.gz', 'rt') as handle:
        records = SeqIO.to_dict(SeqIO.parse(handle, 'fasta'))
        # parse out the ENSG from the keys
        records = {
            ensg.search(name).group(1): record for name, record in records.items()
        }
        
    def extract_sequence(row):
        position = row['position']
        return str(records[row['gene']].seq[position:position+3])
        
    pause_sites['sequence'] = pause_sites.apply(extract_sequence, axis='columns')
    return pause_sites

def find_runs():
    ensg = re.compile(r'(ENSG\d+)')
        
    with gzip.open('/path/to/appris_human_v2_selected.fa.gz', 'rt') as handle:
        records = SeqIO.to_dict(SeqIO.parse(handle, 'fasta'))
        # parse out the ENSG from the keys
        records = {
            ensg.search(name).group(1): record 
            for name, record in records.items()
        }
    result = []
    for id, record in records.items():
    
        # get CDS
        match = re.search(r'CDS:([^|]+)', record.name)
        start, end = match.group(1).split('-')
        start, end = int(start) - 1, int(end)
        aa_seq = record.seq[start:end].translate()
    
        for match in re.finditer(r"[KR]{3,}", str(aa_seq)):
            result.append({
                'GeneID': id,
                'codon': match.start(),
                'seq': match.group(0),
                'length': len(match.group(0)),
            })
    return pd.DataFrame(result)

def calculate_occupancy(pause_sites, gene_coverage, carry_over=['diffTE']):
    occupancy = []
    for _, row in pause_sites.iterrows():
        read_data = gene_coverage.query(
            'gene == @row.gene and codon >= @row.codon - 20 and codon <= @row.codon + 20'
        ).copy()
        group_counts = read_data.groupby(['experiment'])['count']
        # skip reads < 0.5 coverage
        if (group_counts.mean() < 0.5).any():
            continue
        read_data['relative_counts'] = group_counts.transform(lambda x: x / x.mean())
        read_data['relative_codon'] = read_data['codon'] - row.codon
        read_data[carry_over] = row[carry_over]
    
        occupancy.append(read_data)
    
    return pd.concat(occupancy, ignore_index=True)

### S17

In [ ]:
gene_list = pd.concat([
    pd.read_excel('S17/S17monosome_Downregulated_FC1point2_02132025.xlsx').assign(diffTE='down'),
    pd.read_excel('S17/S17monosome_NONregulated_FC1point2_02132025.xlsx').assign(diffTE='none'),
    pd.read_excel('S17/S17monosome_Upregulated_FC1point2_02132025.xlsx').assign(diffTE='up'),
])
gene_list.head()

In [ ]:
gene_coverage = read_coverage({
    exp: f'S17/{exp}_coverage.csv.gz'
    for exp in ('S17T0-1', 'S17T0-2', 'S17T24-1', 'S17T24-2', 'S17T72-1', 'S17T72-2','S17T72-3',)
}, gene_list)
gene_coverage.head()

In [ ]:
pauses = calculate_pause_scores(gene_coverage)
pauses = pauses.merge(gene_list[['GeneID', 'diffTE']], left_on='gene', right_on='GeneID')
g = sns.displot(data=pauses, x='pause_score', hue='experiment', kind='ecdf', col='diffTE')
g.set(xscale="log", xlim=(0.05, 50))

In [ ]:
pause_sites = identify_age_dependent_pauses(gene_coverage, 'S17T0', 'S17T72', 'S17T24', only_significant=True)
pause_sites = pause_sites.merge(gene_list[['GeneID', 'diffTE']], left_on='gene', right_on='GeneID')
pause_sites = add_codons(pause_sites)
sns.relplot(data=pause_sites, x='log2_odds_ratio', y='-log10_adjusted_p_value', col='diffTE')

In [ ]:
occupancy = calculate_occupancy(pause_sites, gene_coverage)
sns.relplot(data=occupancy, x='relative_codon', y='relative_counts', 
            col='experiment', hue='experiment', kind='line', row='diffTE')

In [ ]:
L10A_gene_list =  pd.concat([
    pd.read_excel('L10A/L10Amonosome_ALL-DOWNegulated_Ranked-FC1point2.xlsx').assign(diffTE='down'),
    pd.read_excel('L10A/L10Amonosome_ALL-UPregulated_Ranked-FC1point2.xlsx').assign(diffTE='up'),
])
L10A_gene_list = L10A_gene_list[L10A_gene_list.Rank <= 200]
L10A_gene_list.head()

In [ ]:
gene_coverage = read_coverage({
    exp: f'Protac/{exp}_coverage.csv.gz'
    for exp in ('ENT-1', 'ENT-2', 'HP3-1', 'HP3-2')
}, L10A_gene_list)
gene_coverage.head()

In [ ]:
pauses = calculate_pause_scores(gene_coverage)
pauses = pauses.merge(L10A_gene_list[['GeneID', 'diffTE']], left_on='gene', right_on='GeneID')
pauses.to_csv('Protac/pause_scores_L10A.csv.gz')
g = sns.displot(data=pauses, x='pause_score', hue='experiment', kind='ecdf', col='diffTE')
g.set(xscale="log", xlim=(0.05, 50))
pauses.head()

In [ ]:
with pd.option_context('mode.use_inf_as_na', True):
    top_bottom = pd.concat([
        pd.read_excel('Protac/TL-UPTL-DOWN_rpkm.xlsx', sheet_name='TL-UP').assign(diffTE='up').dropna().head(200),
        pd.read_excel('Protac/TL-UPTL-DOWN_rpkm.xlsx', sheet_name='TL-DOWN').assign(diffTE='down').dropna().head(200),
    ])
top_bottom.head()

In [ ]:
gene_coverage = read_coverage({
    exp: f'Protac/{exp}_coverage.csv.gz'
    for exp in ('ENT-1', 'ENT-2', 'HP3-1', 'HP3-2')
}, top_bottom)
gene_coverage.head()

In [ ]:
pauses = calculate_pause_scores(gene_coverage)
pauses = pauses.merge(top_bottom[['GeneID', 'diffTE']], left_on='gene', right_on='GeneID')
pauses.to_csv('Protac/pause_scores_topbottom.csv.gz')
g = sns.displot(data=pauses, x='pause_score', hue='experiment', kind='ecdf', col='diffTE')
g.set(xscale="log", xlim=(0.05, 50))
pauses.head()

In [ ]:
pause_sites = identify_age_dependent_pauses(gene_coverage, 'ENT', 'HP3', mid_exp=None, only_significant=True)
pause_sites = pause_sites.merge(top_bottom[['GeneID', 'diffTE']], left_on='gene', right_on='GeneID')
pause_sites = add_codons(pause_sites)
pause_sites.to_csv('Protac/pause_sites_topbottom.csv.gz')
sns.relplot(data=pause_sites, x='log2_odds_ratio', y='-log10_adjusted_p_value', col='diffTE')

In [ ]:
occupancy = calculate_occupancy(pause_sites, gene_coverage)
occupancy.to_csv('Protac/ribosome_occupancy_topbottom.csv.gz')
sns.relplot(data=occupancy, x='relative_codon', y='relative_counts', 
            col='experiment', hue='experiment', kind='line', row='diffTE')

In [ ]:
gene_list = pd.concat([
    pd.read_excel('Protac/TL-UPTL-DOWN_rpkm.xlsx', sheet_name='TL-UP').assign(diffTE='up'),
    pd.read_excel('Protac/TL-UPTL-DOWN_rpkm.xlsx', sheet_name='TL-DOWN').assign(diffTE='down'),
    pd.read_excel('Protac/TL-UPTL-DOWN_rpkm.xlsx', sheet_name='NON').assign(diffTE='none'),
])
gene_list.head()

In [ ]:
gene_coverage = read_coverage({
    exp: f'Protac/{exp}_coverage.csv.gz'
    for exp in ('ENT-1', 'ENT-2', 'HP3-1', 'HP3-2')
}, gene_list)
gene_coverage.head()

In [ ]:
pauses = calculate_pause_scores(gene_coverage)
pauses = pauses.merge(gene_list[['GeneID', 'diffTE']], left_on='gene', right_on='GeneID')
pauses.to_csv('Protac/pause_scores.csv.gz')
g = sns.displot(data=pauses, x='pause_score', hue='experiment', kind='ecdf', col='diffTE')
g.set(xscale="log", xlim=(0.05, 50))
pauses.head()

In [ ]:
pause_sites = identify_age_dependent_pauses(gene_coverage, 'ENT', 'HP3', mid_exp=None, only_significant=True)
pause_sites = pause_sites.merge(gene_list[['GeneID', 'diffTE']], left_on='gene', right_on='GeneID')
pause_sites = add_codons(pause_sites)
pause_sites.to_csv('Protac/pause_sites.csv.gz')
sns.relplot(data=pause_sites, x='log2_odds_ratio', y='-log10_adjusted_p_value', col='diffTE')

In [ ]:
occupancy = calculate_occupancy(pause_sites, gene_coverage)
occupancy.to_csv('Protac/ribosome_occupancy.csv.gz')
sns.relplot(data=occupancy, x='relative_codon', y='relative_counts', 
            col='experiment', hue='experiment', kind='line', row='diffTE')

In [ ]:
data = pd.read_csv('/scratch/gpfs/tcomi/ribo_testing/Protac/pause_sites_non_sig.csv.gz')

significant = data[data.adjusted_p_value < 0.05]
print(len(significant))
significant.head()

In [ ]:
from Bio.Seq import Seq

ensg = re.compile(r'(ENSG\d+)')
    
with gzip.open('/scratch/gpfs/tcomi/references_for_riboflow/transcriptome/human/v2/riboflow_annot_and_ref/appris_human_v2_selected.fa.gz', 'rt') as handle:
    records = SeqIO.to_dict(SeqIO.parse(handle, 'fasta'))
    # parse out the ENSG from the keys
    records = {
        ensg.search(name).group(1): record for name, record in records.items()
    }
        
def extract_upstream_sequence(gene, position):
    start = max(0, (position - 29))
    end = position + 4
    return str(records[gene].seq[start-1:end-1])

In [ ]:
result = []
# for sites with significant p value
for index, row in data[data.adjusted_p_value < 0.05].iterrows():
    # get the sequences 10 codons before and including this site
    upstream = extract_upstream_sequence(row['gene'], row['position'])
    row['nn_sequence'] = upstream
    row['aa_sequence'] = str(Seq(upstream).translate())
    result.append(row)
result = pd.DataFrame(result)
result.head()

In [ ]:
with open('Protac/peptides.txt', 'w') as outfile:
    for seq in result['aa_sequence']:
        outfile.write(seq+'\n')

## Polybasic sites

In [ ]:
basic_sites = find_runs()

occupancy = calculate_occupancy(
    basic_sites.rename(columns={'GeneID': 'gene'}),
    gene_coverage, carry_over=['seq', 'length'])
occupancy['capped_length'] = occupancy['length']
occupancy.loc[occupancy['length'] > 6, 'capped_length'] = 6

In [ ]:
sns.relplot(data=occupancy, x='relative_codon', y='relative_counts', col='experiment', hue='experiment', kind='line', row='capped_length')